# 🎓 Ciência de Dados — Aula 13: Aprendizado Supervisionado 1 (Classificação, Fronteiras de Decisão e Arquitetura Híbrida)
**Professor:** Dr. Rafael Duarte  
**Data:** 05 de Novembro de 2026  
**Formato da Aula:** *Live Coding & Literate Programming* (100% guiada via Notebook)

---

### ⏱️ Cronograma e Dinâmica da Aula de Hoje
* **Momento 0 (Primeiros 25 min):** **Banca Presencial do Checkpoint 1 (Equipes Remanescentes da Aula 12).**  
  * *Enquanto passo nas mesas dos grupos que faltaram apresentar na última aula, os demais grupos já devem abrir seus notebooks do projeto e preparar a variável alvo categórica para o laboratório de hoje.*
  * ⚠️ **Lembrete da Regra do Arquiteto (Defesa de Código):** Nos Checkpoints presenciais, **não basta mostrar o código executando na tela**. Cada integrante será questionado sobre *o porquê* das decisões tomadas, o significado das variáveis, como evitou *Data Leakage* e o que as métricas significam para o negócio!
* **Módulo 1:** Da Reta ao "S" da Probabilidade — Por que Regressão Linear quebra em decisões binárias ($0$ ou $1$) e a Matemática da **Regressão Logística** (Função Sigmoide, *Odds* e *Log-Odds*).
* **Módulo 2:** As 3 Escolas de Classificação na Prática (20 Clientes da Fintech *"CrediDados"*): **Regressão Logística** vs. **KNN** (*Vizinhos Mais Próximos*) vs. **Árvore de Decisão** (Impureza de Gini).
* **Módulo 3:** Auditoria de Risco e **Matriz de Confusão** — O Paradoxo da Acurácia, Falsos Positivos vs. Falsos Negativos e o dilema executivo entre **Precisão, Recall e F1-Score**.
* **☕ Intervalo (15 min)**
* **Módulo 4:** **Arquitetura Híbrida (*Mixed Approach*):** Como combinar *Clustering* (Não Supervisionado) + *Propensão ao Risco* (Supervisionado) para criar motores inteligentes de priorização de filas em operações reais!
* **Módulo 5:** Regras e Rubricas do **Checkpoint 2 (12/11)** e da **Entrega Final (25/11)** + Laboratório Prático da Aula 13.

---
# 📘 MÓDULO 1: Da Reta Infinita à Probabilidade (Regressão Logística)

## 1.1. A Fronteira Conceitual: Regressão vs. Classificação Supervisionada
Até aqui, vimos dois grandes paradigmas:
1. **Aprendizado Supervisionado de Regressão (Aula 10):** Tínhamos um gabarito numérico contínuo ($y \in \mathbb{R}$, como prever o `Faturamento` em R$ ou o `Custo de Entrega`).
2. **Aprendizado Não Supervisionado de Clustering (Aulas 11 e 12):** Não tínhamos gabarito $y$; agrupávamos clientes pela proximidade geométrica na matriz $X$.

Hoje entramos no **Aprendizado Supervisionado de Classificação**: nós **temos um gabarito histórico $y$**, mas ele **não é um número contínuo**, e sim uma **Categoria / Classe Discreta**! No caso mais clássico dos negócios (**Classificação Binária**), $y \in \{0, 1\}$:
* $y = 1$ (**Classe Positiva / Evento de Interesse**): Cliente cancelou (*Churn*), Transação é Fraude, Paciente tem a doença, Entrega atrasou, Inquilino quebrou o contrato.
* $y = 0$ (**Classe Negativa / Caso Normal**): Cliente permaneceu ativo, Transação legítima, Paciente saudável, Entrega no prazo.

---

## 1.2. Por que não usar a Regressão Linear da Aula 10 para prever $0$ ou $1$?
Se tentarmos ajustar uma reta comum $\hat{y}_i = b_0 + b_1 x_i$ quando os dados reais só valem $0$ ou $1$, cometemos dois pecados matemáticos graves:
1. **Previsões fora do domínio da Probabilidade $[0, 1]$:** A reta é infinita! Para valores baixos de $x$, ela prevê probabilidades negativas (ex: $-35\%$ de chance de calote); para valores altos de $x$, prevê probabilidades acima de $100\%$ (ex: $+140\%$).
2. **Deformação do Limiar de Corte (*Threshold*) por Outliers:** Um único cliente com renda ou dívida extrema inclina a reta inteira e destrói a fronteira de decisão de $50\%$ ($0,5$).

### A Solução: A Função Sigmoide (Logística)
Em vez de prever $y$ com uma reta sem limites, a **Regressão Logística** (atenção: apesar do nome histórico levar "Regressão", ela é um **algoritmo de Classificação!**) calcula primeiro uma combinação linear $z_i$ e aplica sobre ela a **Função Sigmoide ($\sigma$)**, que curva a reta em formato de **"S"** e garante que o resultado $\hat{p}_i$ fique sempre estritamente entre $0$ e $1$ ($0\%$ a $100\%$):

$$z_i = b_0 + b_1 x_{i1} + b_2 x_{i2} + \dots + b_m x_{im}$$

$$\hat{p}_i = P(y_i = 1 \mid x_i) = \sigma(z_i) = \frac{1}{1 + e^{-z_i}} \quad \Longleftrightarrow \quad \ln\left(\frac{\hat{p}_i}{1 - \hat{p}_i}\right) = z_i$$

<small>
📖 <b>Guia de Leitura Matemática e Notação (População vs. Amostra):</b><br>
• <b>$\hat{p}_i$ vs. $\pi_i$ (Lê-se <i>"pê chapéu índice i"</i> vs. <i>"pi índice i"</i>):</b> Na inferência estatística, a letra grega $\pi_i$ representa a probabilidade verdadeira (e desconhecida) na <b>População</b> de o indivíduo $i$ pertencer à classe $1$; já $\hat{p}_i$ é a probabilidade estimada pelo nosso modelo a partir da <b>Amostra</b>.<br>
• <b>$z_i$ (Lê-se <i>"zê índice i"</i> — Preditor Linear ou <i>Logit</i>):</b> Resultado da soma ponderada das variáveis amostrais $x_{ij}$ multiplicadas pelos pesos amostrais $b_j$ (que estimam os parâmetros populacionais $\beta_j$).<br>
• <b>$\sigma(z_i)$ (Letra grega <i>Sigma</i> minúscula — lê-se <i>"sigma de zê i"</i>):</b> Representa a função de ativação Sigmoide (não confundir com o desvio-padrão populacional $\sigma$).<br>
• <b>$e^{-z_i}$ (Lê-se <i>"número de Euler ê elevado a menos zê i"</i>):</b> Onde $e \approx 2,71828$. Note a mágica: quando $z_i = 0$, temos $e^0 = 1 \implies \hat{p}_i = \frac{1}{1+1} = \mathbf{0,50}$ ($50\%$, exatamente em cima do muro!). Se $z_i > 0$, $\hat{p}_i > 0,50$ (classificamos como $\hat{y}_i = 1$). Se $z_i < 0$, $\hat{p}_i < 0,50$ (classificamos como $\hat{y}_i = 0$).<br>
• <b>$\ln\left(\frac{\hat{p}_i}{1 - \hat{p}_i}\right)$ (Lê-se <i>"logaritmo natural da razão de chances"</i> ou <i>Log-Odds</i>):</b> A fração $\frac{\hat{p}}{1-\hat{p}}$ é a <b>Chance (<i>Odds</i>)</b> — a razão entre a probabilidade de acontecer e a de não acontecer.
</small>

> 🏛️ **Contexto Histórico & Etimológico:**  
> A curva logística foi deduzida em **1838** pelo matemático belga **Pierre François Verhulst** para descrever como populações crescem rapidamente no início e depois saturam no teto de recursos ambientais. Em **1944**, o médico **Joseph Berkson** batizou a unidade logarítmica de **Logit**, e em **1958** o estatístico britânico **Sir David Cox** formulou a Regressão Logística moderna para análise de sobrevivência e risco clínico.

## 1.3. Caso Prático Visível: 20 Clientes da Fintech "CrediDados"
Vamos carregar uma amostra enxuta e totalmente visível de **20 solicitações de crédito (`Cli_01` a `Cli_20`)**, onde queremos prever a variável alvo binária **`Inadimplente_Alvo` ($y$)** a partir de duas variáveis explicativas ($X$):
* `Comprometimento_Renda_Pct` ($x_1$): Percentual da renda mensal já comprometida com parcelas (de $12\%$ a $88\%$).
* `Score_Credito` ($x_2$): Pontuação de bureau de crédito (de $280$ a $910$ pontos).
* **`Inadimplente_Alvo` ($y$):** `0` = Pagou o empréstimo em dia (10 clientes) | `1` = Deu calote / Inadimplente (10 clientes).

Execute a célula abaixo para ver a tabela e comparar o desastre da Reta Linear contra a perfeição da Curva Sigmoide!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

pd.set_option('display.max_rows', 25)
pd.set_option('display.width', 1000)
plt.rcParams['figure.figsize'] = (14, 5.5)
sns.set_theme(style='whitegrid')

# Base visível de 20 clientes (10 Adimplentes = 0 | 10 Inadimplentes = 1)
dados_credito = {
    'ID_Cliente': [f'Cli_{i:02d}' for i in range(1, 21)],
    'Comprometimento_Renda_Pct': [
        12, 15, 18, 22, 25, 28, 32, 35, 38, 42,  # Clientes com dívida controlada
        45, 48, 52, 55, 60, 65, 70, 75, 82, 88   # Clientes altamente endividados
    ],
    'Score_Credito': [
        910, 880, 840, 790, 760, 720, 690, 640, 610, 540,
        590, 520, 480, 450, 410, 390, 350, 320, 295, 280
    ],
    'Inadimplente_Alvo': [
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1,  # Note que o Cli_10 (42% renda, Score 540) deu calote!
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1   # Note que o Cli_11 (45% renda, Score 590) pagou em dia!
    ]
}

df_cred = pd.DataFrame(dados_credito)
display(df_cred)

# Comparando Regressão Linear vs. Regressão Logística usando 'Comprometimento_Renda_Pct'
X_1d = df_cred[['Comprometimento_Renda_Pct']]
y_alvo = df_cred['Inadimplente_Alvo']

reg_lin = LinearRegression().fit(X_1d, y_alvo)
reg_log = LogisticRegression().fit(X_1d, y_alvo)

# Criando grade suave de 0% a 100% de comprometimento de renda para desenhar as curvas
x_grid = pd.DataFrame({'Comprometimento_Renda_Pct': np.linspace(0, 100, 300)})
y_reta = reg_lin.predict(x_grid)
y_sigmoide = reg_log.predict_proba(x_grid)[:, 1] # Coluna 1 = Probabilidade de y=1 (Calote)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

# Painel 1: A Reta Linear vazando fora de [0, 1]
axes[0].scatter(df_cred['Comprometimento_Renda_Pct'], y_alvo, c=y_alvo, cmap='bwr', s=90, edgecolors='k', zorder=5, label='Clientes Reais (0 ou 1)')
axes[0].plot(x_grid['Comprometimento_Renda_Pct'], y_reta, color='crimson', lw=2.5, label='Reta Linear (Vaza p/ <0% e >100%!)')
axes[0].axhline(0, color='gray', linestyle=':')
axes[0].axhline(1, color='gray', linestyle=':')
axes[0].axhline(0.5, color='black', linestyle='--', alpha=0.6, label='Corte 50% (0.5)')
axes[0].set_title('[ERRO CONCEITUAL] Regressão Linear em Alvo Binário')
axes[0].set_xlabel('% da Renda Comprometida')
axes[0].set_ylabel('Previsão Bruta')
axes[0].legend()

# Painel 2: A Curva Sigmoide da Regressão Logística
axes[1].scatter(df_cred['Comprometimento_Renda_Pct'], y_alvo, c=y_alvo, cmap='bwr', s=90, edgecolors='k', zorder=5, label='Clientes Reais (0 ou 1)')
axes[1].plot(x_grid['Comprometimento_Renda_Pct'], y_sigmoide, color='royalblue', lw=3, label='Curva Sigmoide P(y=1 | x)')
axes[1].axhline(0.5, color='darkgreen', linestyle='--', lw=2, label='Threshold de Decisão (p = 0.50)')
axes[1].set_title('[CORRETO] Regressão Logística (Função Sigmoide entre 0 e 1)')
axes[1].set_xlabel('% da Renda Comprometida')
axes[1].set_ylabel('Probabilidade Estimada de Calote (p_hat)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# 🌳 MÓDULO 2: Três Escolas de Pensamento em Classificação (Logística vs. KNN vs. Árvore)

Na prática profissional, nunca testamos um único algoritmo. Vamos comparar como **três famílias clássicas de classificadores** tomam decisões usando as nossas duas variáveis (`Comprometimento_Renda_Pct` e `Score_Credito`):

### 2.1. Regressão Logística (`LogisticRegression`) — Fronteira Linear de Probabilidade
* Traça uma reta diagonal no espaço 2D onde a probabilidade de calote é exatamente $50\%$ ($\hat{p}_i = 0,50$). De um lado da reta estão os aprovados ($\hat{p}_i < 0,50$); do outro, os reprovados ($\hat{p}_i \ge 0,50$).

### 2.2. K-Vizinhos Mais Próximos (`KNeighborsClassifier` — KNN) — Geometria de Proximidade
* Criado por **Evelyn Fix e Joseph Hodges (1951)** para a Força Aérea Americana.
* **Como funciona:** Quando chega um cliente novo, o KNN calcula a **Distância Euclidiana** dele até todos os clientes históricos, seleciona os **$K$ vizinhos mais próximos** (ex: $K=3$) e faz uma votação: se 2 dos 3 vizinhos deram calote, o novo cliente é classificado como caloteiro!
* ⚠️ **Regra Obrigatória:** Como usa Distância Euclidiana (igual ao K-Means da Aula 11!), o KNN **exige padronização prévia com `StandardScaler`**, senão o `Score_Credito` (centenas) esmaga o `Comprometimento_Renda_Pct` (dezenas)!

### 2.3. Árvore de Decisão (`DecisionTreeClassifier` — CART) — Regras Lógicas e Impureza de Gini
* Sistematizada por **Leo Breiman et al. (1984)**, a Árvore de Decisão cria um fluxograma de perguntas do tipo `SE Score_Credito <= 565 ENTÃO...`.
* **Como o algoritmo escolhe a melhor pergunta para dividir um nó?** Ele testa todos os cortes possíveis e escolhe aquele que gera os nós filhos mais **puros** (onde quase todo mundo é só $0$ ou só $1$), minimizando o **Índice de Impureza de Gini** (Corrado Gini, 1912):

$$G(t) = 1 - \sum_{c=0}^{C-1} \hat{p}_{tc}^2 = 1 - \left(\hat{p}_{t0}^2 + \hat{p}_{t1}^2\right)$$

<small>
📖 <b>Guia de Leitura Matemática e Notação:</b><br>
• <b>$G(t)$ (Lê-se <i>"Gini do nó tê"</i>):</b> Grau de impureza (mistura de classes) dentro da caixinha (nó) $t$ da árvore.<br>
• <b>$C$ (Lê-se <i>"cê maiúsculo"</i>):</b> Número total de classes do problema (na classificação binária, $C = 2$: classe $c=0$ e classe $c=1$).<br>
• <b>$\hat{p}_{tc}$ (Lê-se <i>"pê chapéu índice tê, cê"</i>):</b> Proporção amostral de indivíduos da classe $c$ dentro do nó $t$ ($\hat{p}_{tc} = \frac{n_{tc}}{n_t}$).<br>
• <b>Termômetro de Gini:</b><br>
  - Se um nó tem $10$ adimplentes ($\hat{p}_0 = 0,5$) e $10$ caloteiros ($\hat{p}_1 = 0,5$), a mistura é máxima: $G = 1 - (0,5^2 + 0,5^2) = 1 - 0,50 = \mathbf{0,50}$ (Impureza Máxima!).<br>
  - Se após uma pergunta da árvore um nó fica com $0$ adimplentes ($\hat{p}_0 = 0,0$) e $9$ caloteiros ($\hat{p}_1 = 1,0$), temos: $G = 1 - (0^2 + 1^2) = \mathbf{0,00}$ (<b>Nó 100% Puro!</b> A árvore para de dividir e crava a resposta!).
</small>

In [ ]:
# Treinando e comparando os 3 Classificadores na base dos 20 clientes:
X_2d = df_cred[['Comprometimento_Renda_Pct', 'Score_Credito']]
scaler_cred = StandardScaler()
X_2d_scaled = scaler_cred.fit_transform(X_2d)

# 1. Regressão Logística
mod_log = LogisticRegression().fit(X_2d_scaled, y_alvo)
df_cred['Prob_Calote_Logistica'] = mod_log.predict_proba(X_2d_scaled)[:, 1].round(3)
df_cred['Pred_Logistica'] = mod_log.predict(X_2d_scaled)

# 2. KNN (K = 3 vizinhos mais próximos)
mod_knn = KNeighborsClassifier(n_neighbors=3).fit(X_2d_scaled, y_alvo)
df_cred['Pred_KNN_3'] = mod_knn.predict(X_2d_scaled)

# 3. Árvore de Decisão (Profundidade máxima = 2 níveis para evitar decorar ruído / Overfitting)
mod_tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_2d, y_alvo) # Árvore pode usar X na escala original R$/Score!
df_cred['Pred_Arvore'] = mod_tree.predict(X_2d)

print("🔍 Inspecione a zona de fronteira (Clientes 08 a 13) e veja a probabilidade calculada e o voto de cada algoritmo:")
display(df_cred.iloc[7:13])

# Plotando a Árvore de Decisão aprendida pelo critério de Gini!
plt.figure(figsize=(12, 5))
plot_tree(
    mod_tree,
    feature_names=['Comprometimento_Renda_Pct', 'Score_Credito'],
    class_names=['0_Adimplente', '1_Caloteiro'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Árvore de Decisão Aprendida (Observe o valor de "gini" caindo de 0.50 na raiz até ~0.0 nas folhas!)')
plt.tight_layout()
plt.show()

---
# 🎯 MÓDULO 3: Matriz de Confusão e o Dilema Executivo (Precisão vs. Recall)

Como sabemos qual dos 3 classificadores é o melhor para ir para produção?  
Em problemas de Classificação, olhar apenas para a **Acurácia (% de acertos globais)** é um erro primário!

### 3.1. O Paradoxo da Acurácia em Eventos Raros
Imagine uma operadora de cartões onde **99% das transações são legítimas ($0$)** e apenas **1% é fraude ($1$)**.  
Se um programador preguiçoso criar um modelo que simplesmente responde *"Nenhuma transação é fraude (Tudo = 0)"*, esse modelo terá **99% de Acurácia**, mas deixará **100% dos fraudadores roubarem o banco**!

Para auditar a anatomia dos acertos e dos erros de um classificador, construímos a **Matriz de Confusão**:

| | **Modelo Previu: $0$ (Negativo)** | **Modelo Previu: $1$ (Positivo)** |
| :--- | :---: | :---: |
| **Gabarito Real: $0$ (Negativo)** | ✅ **Verdadeiro Negativo ($VN$)**<br>*(Era bom pagador e o modelo aprovou)* | ❌ **Falso Positivo ($FP$ — Erro Tipo I)**<br>*(Alarme Falso: era bom cliente, mas o modelo barrou!)* |
| **Gabarito Real: $1$ (Positivo)** | 🚨 **Falso Negativo ($FN$ — Erro Tipo II)**<br>*(Omissão Crítica: era caloteiro/fraude e o modelo deixou passar!)* | ✅ **Verdadeiro Positivo ($VP$)**<br>*(Era caloteiro/fraude e o modelo bloqueou com sucesso!)* |

---

### 3.2. As 4 Métricas de Ouro da Classificação

$$\text{Acurácia} = \frac{VP + VN}{n_{\text{amostra}}} \qquad \text{Precisão (Precision)} = \frac{VP}{VP + FP} \qquad \text{Revocação (Recall)} = \frac{VP}{VP + FN}$$

$$F_1\text{-Score} = 2 \cdot \frac{\text{Precisão} \cdot \text{Recall}}{\text{Precisão} + \text{Recall}}$$

<small>
📖 <b>Guia de Leitura Matemática e Qual Métrica Escolher no Projeto do Seu Grupo:</b><br>
• <b>$VP, VN, FP, FN$ (Lê-se <i>"Verdadeiro Positivo, Verdadeiro Negativo, Falso Positivo, Falso Negativo"</i>):</b> Contagens inteiras de indivíduos na amostra avaliada (em inglês: $TP, TN, FP, FN$). A soma dos quatro é igual ao tamanho da amostra testada ($n_{\text{amostra}}$).<br>
• <b>Precisão ($\frac{VP}{VP + FP}$ — Foco na Certeza do Ataque):</b> Responde: <i>"De todos os clientes que o meu modelo acusou como positivos ($1$), qual porcentagem realmente era positiva?"</i>. Use quando o <b>Falso Positivo ($FP$) for muito caro ou constrangedor</b> (ex: bloquear o cartão do cliente VIP na fila do restaurante, ou jogar um e-mail de trabalho importante na caixa de Spam).<br>
• <b>Recall / Revocação / Sensibilidade ($\frac{VP}{VP + FN}$ — Foco na Rede de Proteção):</b> Responde: <i>"De todos os casos positivos reais ($1$) que estavam escondidos na base, qual porcentagem o meu modelo conseguiu capturar?"</i>. Use quando o <b>Falso Negativo ($FN$) for catastrófico ou fatal</b> (ex: deixar passar um tumor maligno em exame médico, deixar passar uma trinca em turbina de avião, ou emprestar R$ 500 mil para um fraudador).<br>
• <b>$F_1\text{-Score}$ (Média Harmônica):</b> Equilibra Precisão e Recall em um único número entre $0$ e $1$. Só é alto se AMBAS as métricas forem altas!
</small>

In [ ]:
# Avaliando a Matriz de Confusão e as Métricas da nossa Regressão Logística:
cm = confusion_matrix(y_alvo, df_cred['Pred_Logistica'])
vn, fp, fn, vp = cm.ravel()

print(f"Desmembrando a Matriz de Confusão: VN = {vn} | FP (Alarme Falso) = {fp} | FN (Calote que passou!) = {fn} | VP = {vp}\n")
print("Relatório Oficial de Classificação (Scikit-Learn):")
print(classification_report(y_alvo, df_cred['Pred_Logistica'], target_names=['0_Adimplente', '1_Caloteiro']))

plt.figure(figsize=(6, 4.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={'size': 14, 'weight': 'bold'},
    xticklabels=['Previu 0 (Bom)', 'Previu 1 (Calote)'],
    yticklabels=['Real 0 (Bom)', 'Real 1 (Calote)']
)
plt.title('Matriz de Confusão — Regressão Logística')
plt.ylabel('Realidade (Gabarito y)')
plt.xlabel('Previsão do Modelo (y_hat)')
plt.tight_layout()
plt.show()

---
# ☕ INTERVALO (15 Minutos)
Na volta do intervalo:
1. **Módulo 4:** **A Ótica do *Mixed Approach* (Arquitetura Híbrida):** Como unir *Clustering* (Não Supervisionado) + *Classificação de Propensão* (Supervisionado) para criar sistemas inteligentes de priorização de filas e retenção de contratos!
2. **Módulo 5:** Regras e Rubricas de Avaliação do **Checkpoint 2 (12/11)** e da **Entrega Final (25/11)** + Laboratório Prático da Aula 13.

---
# 🧠 MÓDULO 4: Arquitetura Híbrida (*Mixed Approach*) — Unindo Clustering + Classificação na Vida Real

Muitos alunos perguntam: *"Professor, no meu projeto ou na minha empresa, eu devo usar Aprendizado Não Supervisionado (Clustering) OU Aprendizado Supervisionado (Regressão/Classificação)?"*  

A resposta de um Arquiteto de Soluções sênior é: **Por que não encadear os dois em um Pipeline Híbrido (*Mixed Approach*)?**

### 4.1. O Estudo de Caso "ConectaSul Telecom" (Priorização Inteligente de Chamados de Suporte)
Imagine uma operadora de internet fibra óptica que sofre com o cancelamento de contratos (**Churn**, $y = 1$). A base histórica tem dados do plano (`Valor_Mensalidade_BRL`, `Velocidade_Contratada_Mbps`) e dados operacionais de suporte (`Qtd_Chamados_Suporte_Semestre`, `Tempo_Medio_Espera_Horas`).

Se você olhar todos os clientes misturados numa média geral, você perde uma nuance comportamental crítica:
* **Tribo 1 — Clientes Corporativos / Links Dedicados (Mensalidades de R$ 800 a R$ 1.500):** Têm **tolerância zero a falhas**. Se abrirem **2 chamados** de queda de sinal no semestre e esperarem mais de 2 horas, a probabilidade de cancelarem o contrato (*Churn*) explode para mais de **85%**!
* **Tribo 2 — Clientes Residenciais Econômicos (Planos de R$ 79 a R$ 99):** Por restrição de orçamento ou falta de outra operadora no bairro, eles toleram até **3 ou 4 chamados** simples no semestre sem cancelar (absorvem um atendimento mais lento), só cancelando se o problema passar de 5 ou 6 chamados!

### Como a Arquitetura Híbrida (*Mixed Approach*) Resolve Isso em 3 Passos?
1. **Passo 1 (Não Supervisionado — `KMeans`):** Agrupamos a carteira pelo perfil socioeconômico/contratual (`Valor_Mensalidade` e `Velocidade_Mbps`), descobrindo automaticamente a qual **Cluster de Perfil** cada contrato pertence (`Cluster_Econômico`, `Cluster_Gamer_Intermediário`, `Cluster_Corporativo_VIP`).
2. **Passo 2 (Engenharia de Atributos + Supervisionado — `LogisticRegression` / `Tree`):** Usamos o `Cluster` descoberto no Passo 1 combinado com a `Qtd_Chamados_Suporte` para treinar um **Classificador de Propensão ao Churn**, calculando a probabilidade exata $\hat{p}_i$ de o cliente quebrar o contrato após $1, 2, 3 \dots$ chamados!
3. **Passo 3 (Ação Prescritiva no Sistema de Service Desk):** Quando um cliente abre um chamado técnico hoje, o sistema cruza o **Cluster** dele com o **Nº de Chamados Acumulados** e define automaticamente a **Prioridade da Fila de Atendimento (SLA)**!

Rode a célula abaixo e veja esse *Mixed Approach* funcionando ao vivo!

In [ ]:
# ==============================================================================
# DEMONSTRAÇÃO EXECUTIVA: ARQUITETURA HÍBRIDA (CLUSTERING + PROPENSÃO AO CHURN)
# Caso: Operadora "ConectaSul Telecom" (300 Contratos)
# ==============================================================================
np.random.seed(13)

# 1. Simulando 3 perfis contratuais na base da operadora (300 clientes):
# Perfil Econômico (130 clientes): Mensalidade ~ R$ 90, 200 Mbps (Alta tolerância a chamados: só cancela com >= 5 chamados)
mens_eco = np.random.normal(90, 12, 130).clip(69, 130)
vel_eco  = np.random.normal(200, 30, 130).clip(100, 300)
cham_eco = np.random.poisson(2.2, 130).clip(0, 8)
churn_eco = (cham_eco >= 5).astype(int)

# Perfil Gamer / Home-Office (100 clientes): Mensalidade ~ R$ 190, 600 Mbps (Média tolerância: cancela com >= 3 chamados)
mens_gam = np.random.normal(190, 25, 100).clip(140, 260)
vel_gam  = np.random.normal(600, 60, 100).clip(400, 800)
cham_gam = np.random.poisson(2.0, 100).clip(0, 7)
churn_gam = (cham_gam >= 3).astype(int)

# Perfil Corporativo VIP (70 clientes): Mensalidade ~ R$ 950, 1000 Mbps (Baixíssima tolerância: cancela já no 2º chamado!)
mens_vip = np.random.normal(950, 140, 70).clip(650, 1400)
vel_vip  = np.random.normal(1000, 80, 70).clip(800, 1200)
cham_vip = np.random.poisson(1.5, 70).clip(0, 5)
churn_vip = (cham_vip >= 2).astype(int)

df_telecom = pd.DataFrame({
    'Mensalidade_BRL': np.concatenate([mens_eco, mens_gam, mens_vip]).round(2),
    'Velocidade_Mbps': np.concatenate([vel_eco, vel_gam, vel_vip]).round(0),
    'Chamados_Suporte_6M': np.concatenate([cham_eco, cham_gam, cham_vip]),
    'Quebrou_Contrato_Churn': np.concatenate([churn_eco, churn_gam, churn_vip])
})

# PASSO 1 (NÃO SUPERVISIONADO): Descobrindo o Cluster de Perfil do Contrato (Sem olhar para o Churn!)
X_perfil_sc = StandardScaler().fit_transform(df_telecom[['Mensalidade_BRL', 'Velocidade_Mbps']])
df_telecom['Cluster_Perfil'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_perfil_sc)

# Mapeando os nomes das Personas pela média da mensalidade de cada cluster
mapa_nomes = df_telecom.groupby('Cluster_Perfil')['Mensalidade_BRL'].mean().sort_values().index
nomes_dict = {mapa_nomes[0]: '1_Economico_Residencial', mapa_nomes[1]: '2_Gamer_HomeOffice', mapa_nomes[2]: '3_Corporativo_VIP'}
df_telecom['Tribo_Cliente'] = df_telecom['Cluster_Perfil'].map(nomes_dict)

# PASSO 2 (SUPERVISIONADO POR FATIA): Treinando Regressão Logística para medir a Curva de Tolerância a Chamados por Tribo!
plt.figure(figsize=(11, 5))
cores_tribo = {'1_Economico_Residencial': 'seagreen', '2_Gamer_HomeOffice': 'darkorange', '3_Corporativo_VIP': 'crimson'}
x_chamados_grid = pd.DataFrame({'Chamados_Suporte_6M': np.linspace(0, 7, 100)})

for nome_t, cor in cores_tribo.items():
    sub_df = df_telecom[df_telecom['Tribo_Cliente'] == nome_t]
    mod_tol = LogisticRegression().fit(sub_df[['Chamados_Suporte_6M']], sub_df['Quebrou_Contrato_Churn'])
    prob_curva = mod_tol.predict_proba(x_chamados_grid)[:, 1] * 100
    plt.plot(x_chamados_grid, prob_curva, lw=3, color=cor, label=f'Curva de Risco: {nome_t}')

plt.axhline(50, color='black', linestyle='--', alpha=0.6, label='Gatilho Crítico de Risco (50% de Chance de Quebra)')
plt.title('Mixed Approach: Curva de Tolerância a Chamados de Suporte por Cluster de Cliente')
plt.xlabel('Quantidade de Chamados de Manutenção/Suporte Abertos no Semestre')
plt.ylabel('Probabilidade Estimada de Quebra de Contrato / Churn (%)')
plt.xticks(range(0, 8))
plt.legend()
plt.tight_layout()
plt.show()

print("💡 Aplicação Prática no Sistema de Chamados:")
print("   - Se um cliente '3_Corporativo_VIP' abrir o seu 2º chamado -> Risco > 80% -> SLA VERMELHO (Envio imediato de técnico sênior!).")
print("   - Se um cliente '1_Economico_Residencial' abrir o 2º chamado -> Risco < 5%  -> SLA PADRÃO (Atendimento normal na fila de 48h).")

---
# 🚀 MÓDULO 5: Regras do Checkpoint 2 (12/11), Entrega Final (25/11) e Laboratório

## 5.1. Calendário Oficial de Entregas e a "Regra do Arquiteto" (Defesa do Código)

| Etapa Avaliativa | Data Limite | Pontuação Base | Formato e O Que Será Cobrado na Avaliação |
| :--- | :---: | :---: | :--- |
| **Checkpoint 1** *(Remanescentes)* | **Hoje (05/11)** | **1,25 pts** | **Banca Mesa em Mesa:** Wrangling (Aula 09) + Regressão Linear (Aula 10) + Clustering K-Means/DBSCAN/PCA (Aulas 11 e 12). |
| **Checkpoint 2** | **Próxima Aula (12/11)** | **1,25 pts** | **Banca Mesa em Mesa + Blackboard:** Evolução do Notebook com **Classificação Clássica (Aula 13 de hoje) + Ensembles e Redes Neurais (Aula 14)** rodando com Matriz de Confusão e comparação de métricas. |
| **Mentoria Geral (Aula 15)** | **19/11** | *Revisão* | **Sessão Presencial de Tira-Dúvidas:** Ajuste fino dos códigos da Aula 14 e roteirização executiva antes da gravação do vídeo final. |
| **Checkpoint 3 / ENTREGA FINAL** | **25/11 (às 23h59)** | **2,50 pts** | **Vídeo Pitch (5 a 8 min) + Notebook Final Executado + Avaliação por Pares (Soma Zero)** no Blackboard. *(Prazo fatal improrrogável antes das bancas de TCC de 26/11).* |

> ⚖️ **Atenção para a Defesa Oral do Código ("Regra do Arquiteto"):**  
> Tanto no **Checkpoint 1** quanto no **Checkpoint 2** e no **Vídeo Final**, **mostrar o código rodando sem dar erro NÃO garante a nota**.  
> O uso de IA generativa como copiloto de estudo é permitido, mas cada integrante do grupo será arguido individualmente na bancada sobre:
> 1. **Domínio das Variáveis e Preparação:** Por que escolheu tais colunas explicativas ($X$), qual é a variável alvo ($y$), como evitou *Data Leakage* no `train_test_split` e por que aplicou (ou não) o `StandardScaler`.
> 2. **Domínio da Modelagem e Hiperparâmetros:** Como o algoritmo escolhido funciona geometricamente e o que significam os parâmetros passados na função.
> 3. **Leitura Executiva das Métricas:** Saber explicar na ponta da língua se o negócio do grupo sofre mais com **Falso Positivo ($FP$)** ou **Falso Negativo ($FN$)** e defender por que priorizou **Precisão** ou **Recall**.

---

## 5.2. Laboratório de Fixação da Aula 13 (Para Executar Agora nos Últimos 40 Minutos!)

Enquanto finalizo a avaliação presencial das equipes remanescentes na bancada, todos os grupos devem executar o pipeline de **Classificação Supervisionada**:
* **MODALIDADE A (No Dataset Oficial do Projeto do Grupo):** Defina a variável alvo categórica/binária ($y$) do projeto de vocês, separe Treino e Teste com `stratify=y`, aplique `StandardScaler` e compare `LogisticRegression`, `KNeighborsClassifier` e `DecisionTreeClassifier` gerando a Matriz de Confusão!
* **MODALIDADE B (No Dataset Oficial `df_logi_atraso` da LogiTech Brasil abaixo):** Descubra quais pedidos sofrerão **Atraso Crítico (`Atraso_Critico_0_1 = 1`)** que gera multa contratual para a transportadora!

In [ ]:
# ==============================================================================
# GERADOR DA BASE DO LABORATÓRIO DA AULA 13: PREVISÃO DE ATRASO CRÍTICO (LOGITECH)
# ==============================================================================
np.random.seed(2026)
n_ped = 300

dist_km = np.random.gamma(shape=3.0, scale=110.0, size=n_ped).round(1)
peso_kg = np.random.uniform(1.0, 45.0, size=n_ped).round(2)
tempo_sep = np.random.uniform(10.0, 95.0, size=n_ped).round(1)
chuva_rota = np.random.binomial(1, 0.30, size=n_ped) # 0 = Tempo Seco | 1 = Chuva Forte na Rodovia

# Equação latente de risco de atraso (Logit verdadeiro da população):
z_risco = -4.2 + 0.0055 * dist_km + 0.035 * peso_kg + 0.025 * tempo_sep + 1.65 * chuva_rota + np.random.normal(0, 0.6, size=n_ped)
prob_atraso = 1 / (1 + np.exp(-z_risco))
atraso_critico = (prob_atraso >= 0.50).astype(int)

df_logi_atraso = pd.DataFrame({
    'ID_Pedido': [f'PED-{3001+i}' for i in range(n_ped)],
    'Distancia_KM': dist_km,
    'Peso_KG': peso_kg,
    'Tempo_Separacao_Min': tempo_sep,
    'Chuva_na_Rota_0_1': chuva_rota,
    'Atraso_Critico_0_1': atraso_critico
})

print(f"✅ Base 'df_logi_atraso' carregada! Total: {len(df_logi_atraso)} pedidos.")
print("Proporção das classes em 'Atraso_Critico_0_1' (0 = No Prazo | 1 = Atraso com Multa):")
print(df_logi_atraso['Atraso_Critico_0_1'].value_counts(normalize=True).round(3))
display(df_logi_atraso.head(8))

### 📋 Roteiro de Tarefas da Atividade de Fixação (Complete nas 3 células abaixo):

1. **ETAPA 1 — Separação Treino/Teste Estratificada e Padronização sem *Leakage*:**
   * Defina `X = df_logi_atraso[['Distancia_KM', 'Peso_KG', 'Tempo_Separacao_Min', 'Chuva_na_Rota_0_1']]` e `y = df_logi_atraso['Atraso_Critico_0_1']`.
   * Faça o `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)`. *(O parâmetro `stratify=y` garante que a mesma proporção de atrasos caia no Treino e no Teste!).*
   * Ajuste o `StandardScaler()` **apenas no `X_train`** (`scaler.fit_transform(X_train)`) e transforme o `X_test` (`scaler.transform(X_test)`).
2. **ETAPA 2 — Treinamento dos 3 Classificadores Clássicos:**
   * Treine no conjunto de treino padronizado:
     1. `LogisticRegression()`
     2. `KNeighborsClassifier(n_neighbors=5)`
     3. `DecisionTreeClassifier(max_depth=3, random_state=42)`
3. **ETAPA 3 — Matriz de Confusão, Relatório de Métricas e Decisão de Negócio:**
   * Avalie os 3 modelos no conjunto de **Teste (`X_test_scaled`, `y_test`)** imprimindo o `classification_report` e plotando a `confusion_matrix`.
   * Responda em comentário (`#`): **Se cada atraso não previsto (Falso Negativo) gera uma multa pesada para a LogiTech, qual métrica é mais importante monitorar na classe `1` (Precisão ou Recall)? Qual dos 3 modelos teve o melhor desempenho hoje?**

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 1: TRAIN/TEST SPLIT ESTRATIFICADO + STANDARDSCALER
# ==============================================================================
# Escreva seu código aqui:

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 2: TREINAMENTO (LOGÍSTICA, KNN E ÁRVORE DE DECISÃO)
# ==============================================================================
# Escreva seu código aqui:

In [ ]:
# ==============================================================================
# ESPAÇO DO GRUPO — ETAPA 3: MATRIZ DE CONFUSÃO E DEFESA EXECUTIVA DAS MÉTRICAS
# ==============================================================================
# Escreva seu código aqui: